This script takes in the rphi file creates a file without the phi and leaves just the r columns.

In [3]:
import pandas as pd
def process_csv_columns(input_file, output_file, num_columns=192):
    try:
        columns_to_select = list(range(num_columns))
        
        df = pd.read_csv(input_file, usecols=columns_to_select)
        
        df.to_csv(output_file, index=False)
        
        print(f"Successfully processed and saved the first {num_columns} columns to {output_file}")
        
    except FileNotFoundError:
        print(f"Error: The input file '{input_file}' was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

input_csv_file = 'features_polar.csv'
output_csv_file = 'features_justr.csv'

process_csv_columns(input_csv_file, output_csv_file, num_columns=192)

Successfully processed and saved the first 192 columns to features_justr.csv


In [2]:
import pandas as pd
import numpy as np

def normalize_phi_csv(input_path, output_path, n_feat=192):
    """
    Assumes CSV format:
    - first n_feat columns = real (r)
    - next n_feat columns = imag (phi)
    """
    
    df = pd.read_csv(input_path)
    
    # Split into r and phi
    r = df.iloc[:, :n_feat].values
    phi = df.iloc[:, n_feat:].values
    
    # Normalize phi
    phi_norm = phi / (2 * np.pi)
    
    # Recombine
    df_norm = pd.DataFrame(
        np.hstack([r, phi_norm]),
        columns=df.columns
    )
    
    # Save
    df_norm.to_csv(output_path, index=False)
    
    print(f"Saved normalized CSV to: {output_path}")
    print(f"Shape: {df_norm.shape}")

normalize_phi_csv('features_polar.csv', 'features_polar_norm.csv')

Saved normalized CSV to: features_polar_norm.csv
Shape: (111, 384)


In [7]:
import numpy as np
import pandas as pd

def reformat_rphi_csv(input_csv, output_csv):
    """
    Reads a CSV with [real | imag] (or r/phi) style layout,
    converts to [r1 | r2 | dphi], and saves a new CSV.

    Assumes original shape corresponds to:
    (N, 3 TX, 4 SYM, 2 ANT, 8 PKT)
    """

    # constants
    N_TX  = 3
    N_SYM = 4
    N_ANT = 2
    N_PKT = 8

    df = pd.read_csv(input_csv)
    inputs = df.values
    N = inputs.shape[0]

    N_FEAT = N_TX * N_SYM * N_ANT * N_PKT  # 192

    # ---- reconstruct complex (r, phi) ----
    r   = inputs[:, :N_FEAT].reshape(N, N_TX, N_SYM, N_ANT, N_PKT)
    phi = inputs[:, N_FEAT:].reshape(N, N_TX, N_SYM, N_ANT, N_PKT)

    # ---- split antennas ----
    r1 = r[:, :, :, 0, :]  # (N, 3, 4, 8)
    r2 = r[:, :, :, 1, :]  # (N, 3, 4, 8)

    phi1 = phi[:, :, :, 0, :]
    phi2 = phi[:, :, :, 1, :]

    # ---- compute delta phi ----
    dphi = phi2 - phi1

    # optional: normalize to [-0.5, 0.5]
    dphi = dphi / (2 * np.pi)

    # ---- flatten ----
    r1_flat   = r1.reshape(N, -1)   # (N, 96)
    r2_flat   = r2.reshape(N, -1)
    dphi_flat = dphi.reshape(N, -1)

    X_new = np.hstack([r1_flat, r2_flat, dphi_flat])  # (N, 288)

    # ---- column names ----
    n_feat = r1_flat.shape[1]
    cols = (
        [f"r1_{i}" for i in range(n_feat)] +
        [f"r2_{i}" for i in range(n_feat)] +
        [f"dphi_{i}" for i in range(n_feat)]
    )

    df_new = pd.DataFrame(X_new, columns=cols)
    df_new.to_csv(output_csv, index=False)

    print(f"Saved reformatted dataset: {df_new.shape} -> {output_csv}")

reformat_rphi_csv(
    input_csv="features_polar.csv",
    output_csv="features_r1_r2_dphi.csv"
)

Saved reformatted dataset: (111, 288) -> features_r1_r2_dphi.csv
